# AlphaGenome Variant Annotation Workflow

This notebook replaces the active local Borzoi annotation workflow with an AlphaGenome API-based workflow.

It consumes Phase 1 variants from this repo, scores them with AlphaGenome, filters to RNA + lung-like outputs for the target gene, and exports a one-value-per-variant annotation vector.


## Legacy Borzoi Workflow Summary

1. Load GTF-derived gene metadata and exon intervals.
2. Load Borzoi model replicates locally.
3. Filter Borzoi tasks to RNA / GTEx / lung rows.
4. Load the local variant VCF.
5. For each variant, build REF and ALT sequence windows.
6. Run Borzoi predictions for REF and ALT alleles.
7. Aggregate selected RNA tracks over gene exon bins.
8. Compute variant effect as ALT minus REF expression signal.
9. Normalize effects across variants and export the annotation vector.

The AlphaGenome workflow below keeps the same high-level goal but delegates variant scoring to the AlphaGenome API instead of running Borzoi locally.


In [ ]:
import importlib.util
import subprocess
import sys


def ensure_pkg(name: str) -> None:
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', name])


for pkg in ['pandas', 'pyarrow', 'tqdm', 'alphagenome']:
    ensure_pkg(pkg)


In [ ]:
import gc
import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from alphagenome.models import dna_client, variant_scorers

def find_project_root(start: Path | None = None) -> Path:
    candidates = []
    current = (start or Path.cwd()).resolve()
    candidates.extend([current, *current.parents])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate

    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.alphagenome import (
    aggregate_mean_raw_score_by_variant,
    build_run_summary,
    extract_interval_genes,
    filter_scores_to_target_context,
    score_batch,
    tss_radius_bp,
)
from utils.annotations import download_gtf_if_needed, get_gene_info, load_gene_annotations_for_gene
from utils.paths import add_project_root_to_sys_path, configure_runtime_env
from utils.variant_processing import chunked, annotate_variant_window_eligibility, ensure_variant_columns, sanitize_for_parquet

PROJECT_ROOT = add_project_root_to_sys_path()
PATHS = configure_runtime_env(PROJECT_ROOT)
LD_OUTPUT_DIR = PATHS.output_ld
ANNOTATION_OUTPUT_DIR = PATHS.output_annotation_alphagenome
PRELIM_OUTPUT_DIR = PATHS.output_prelim
GTF_CACHE_DIR = PATHS.gtf_cache
GTF_SHORTCUT_DIR = PATHS.gtf_shortcuts

print(f'Project root: {PROJECT_ROOT}')
print(f'LD output dir: {LD_OUTPUT_DIR}')
print(f'Annotation output dir: {ANNOTATION_OUTPUT_DIR}')


In [ ]:
# =============================================================================
# USER INPUTS
# =============================================================================

GENE_NAME = 'CBX8'
GENE_ID = 'ENSG00000141570.11'
REFERENCE_GENOME = 'hg38'

VARIANT_SOURCE_FILE = LD_OUTPUT_DIR / f'{GENE_NAME}_phase1_master_variants.csv'
ALPHAGENOME_API_KEY_ENV_VAR = 'ALPHAGENOME_API_KEY'
SEQUENCE_LENGTH = '1MB'

SCORER_MODE = 'gene_expression'
CENTERING_MODE = 'tss'
TARGET_DATA_SOURCE = 'gtex'
TARGET_GTEX_TISSUE = 'Lung'
BATCH_SIZE = 100
MAX_VARIANTS = None
SCORING_MAX_WORKERS = 8
WRITE_BATCH_CHECKPOINTS = True
RETRY_WAIT_SECONDS = 5

FILTERED_SCORES_PATH = ANNOTATION_OUTPUT_DIR / f'{GENE_NAME}_alphagenome_filtered_scores.parquet'
AGGREGATED_SCORES_PATH = ANNOTATION_OUTPUT_DIR / f'{GENE_NAME}_alphagenome_variant_scores.csv'
ANNOTATION_HISTOGRAM_PATH = ANNOTATION_OUTPUT_DIR / f'{GENE_NAME}_alphagenome_variant_scores_histogram.png'
ANNOTATION_TRIMMED_HISTOGRAM_PATH = ANNOTATION_OUTPUT_DIR / f'{GENE_NAME}_alphagenome_variant_scores_histogram_trimmed.png'
BATCH_SHARD_DIR = ANNOTATION_OUTPUT_DIR / f'{GENE_NAME}_alphagenome_filtered_scores_batches'
INTERVAL_GENES_PATH = PRELIM_OUTPUT_DIR / f'{GENE_NAME}_alphagenome_interval_genes.csv'


In [ ]:
def initialize_batch_shard_dir(shard_dir: Path) -> None:
    if shard_dir.exists():
        shutil.rmtree(shard_dir)
    shard_dir.mkdir(parents=True, exist_ok=True)


def write_filtered_batch_shard(filtered_scores_df: pd.DataFrame, shard_dir: Path, batch_idx: int) -> Path | None:
    if filtered_scores_df.empty:
        return None

    shard_path = shard_dir / f'part-{batch_idx:04d}.parquet'
    sanitize_for_parquet(filtered_scores_df).to_parquet(shard_path, index=False)
    return shard_path


def update_aggregate_state(aggregate_state: dict[str, dict[str, float]], filtered_scores_df: pd.DataFrame) -> None:
    if filtered_scores_df.empty:
        return

    batch_stats = (
        filtered_scores_df.groupby('source_variant_id', as_index=False)
        .agg(raw_score_sum=('raw_score', 'sum'), raw_score_count=('raw_score', 'size'))
    )
    for row in batch_stats.itertuples(index=False):
        state = aggregate_state.setdefault(row.source_variant_id, {'raw_score_sum': 0.0, 'raw_score_count': 0})
        state['raw_score_sum'] += float(row.raw_score_sum)
        state['raw_score_count'] += int(row.raw_score_count)


def finalize_aggregated_scores(aggregate_state: dict[str, dict[str, float]]) -> pd.DataFrame:
    records = []
    for variant_id, state in aggregate_state.items():
        count = int(state['raw_score_count'])
        mean_score = float(state['raw_score_sum']) / count if count else float('nan')
        records.append(
            {
                'source_variant_id': variant_id,
                'alphagenome_raw_mean': mean_score,
                'alphagenome_track_count': count,
            }
        )
    return pd.DataFrame(records).sort_values('source_variant_id').reset_index(drop=True) if records else pd.DataFrame(
        columns=['source_variant_id', 'alphagenome_raw_mean', 'alphagenome_track_count']
    )


def finalize_filtered_scores(shard_dir: Path, output_path: Path) -> int:
    shard_paths = sorted(shard_dir.glob('part-*.parquet'))
    if not shard_paths:
        pd.DataFrame().to_parquet(output_path, index=False)
        return 0

    writer = None
    total_rows = 0
    try:
        for shard_path in shard_paths:
            table = pq.read_table(shard_path)
            total_rows += table.num_rows
            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema, compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    return total_rows


In [ ]:
# Load gene metadata from cached or downloaded GTF resources.
gtf_gene_shortcut = GTF_SHORTCUT_DIR / f'{GENE_NAME}_gene_annotations.csv'
gtf_exon_shortcut = GTF_SHORTCUT_DIR / f'{GENE_NAME}_exon_annotations.csv'
gtf_needed = not (gtf_gene_shortcut.exists() and gtf_exon_shortcut.exists())
if gtf_needed:
    gtf_path = download_gtf_if_needed(GTF_CACHE_DIR, genome=REFERENCE_GENOME)
else:
    gtf_path = GTF_CACHE_DIR / f'{REFERENCE_GENOME}_gencode.gtf.gz'

genes_df, exons_df, used_gtf_shortcut = load_gene_annotations_for_gene(
    gtf_path,
    GENE_NAME,
    GTF_SHORTCUT_DIR,
)
gene_info = get_gene_info(GENE_NAME, genes_df, exons_df)

print(f"Gene: {gene_info['gene_name']}")
print(f"  Location: {gene_info['chrom']}:{gene_info['start']}-{gene_info['end']}")
print(f"  Strand: {gene_info['strand']}")
print(f"  TSS: {gene_info['tss']}")
print(f"  Annotations source: {'shortcut' if used_gtf_shortcut else 'full GTF parse'}")


In [ ]:
raw_variants_df = pd.read_csv(VARIANT_SOURCE_FILE)
raw_variants_df = ensure_variant_columns(raw_variants_df)
raw_variants_df = raw_variants_df[['variant_id', 'chrom', 'pos', 'ref', 'alt', 'z_score']].copy()
variants_loaded_count = len(raw_variants_df)
variants_df = annotate_variant_window_eligibility(raw_variants_df, gene_info, SEQUENCE_LENGTH)
eligible_variants_df = variants_df[~variants_df['excluded_from_scoring']].copy()

if MAX_VARIANTS is not None:
    eligible_variants_df = eligible_variants_df.head(int(MAX_VARIANTS)).copy()

print(f'Loaded {variants_loaded_count} variants from {VARIANT_SOURCE_FILE}')
print(
    f"Eligible variants: total={len(eligible_variants_df)}, "
    f"tss_centered={int((variants_df['scoring_mode'] == 'tss_centered').sum())}, "
    f"midpoint_centered={int((variants_df['scoring_mode'] == 'midpoint_centered').sum())}, "
    f"ineligible={int((variants_df['scoring_mode'] == 'ineligible').sum())}"
)
eligible_variants_df.head()


In [ ]:
api_key = os.environ.get(ALPHAGENOME_API_KEY_ENV_VAR)
if not api_key:
    raise RuntimeError(
        f'Missing AlphaGenome API key. Set {ALPHAGENOME_API_KEY_ENV_VAR} in your shell before running this notebook. '
        'Do not hardcode the key into the notebook source.'
    )

dna_model = dna_client.create(api_key)
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[f'SEQUENCE_LENGTH_{SEQUENCE_LENGTH}']

print('AlphaGenome client initialized.')
print(f'Sequence length: {SEQUENCE_LENGTH}')


In [ ]:
# Gene-expression-first scorer setup.
# If AlphaGenome changes this constructor in a future release, this is the main cell to update.
if SCORER_MODE != 'gene_expression':
    raise ValueError(f'Unsupported SCORER_MODE for this notebook: {SCORER_MODE}')

target_output = dna_client.OutputType.RNA_SEQ
variant_scorer = variant_scorers.GeneMaskLFCScorer(requested_output=target_output)
active_variant_scorers = [variant_scorer]

print(f'Active scorer: {variant_scorer}')


In [ ]:
if eligible_variants_df.empty:
    raise ValueError('No variants remain after TSS-radius filtering; cannot run AlphaGenome preflight.')

preflight_variant_df = eligible_variants_df.head(1).copy()
preflight_full_scores_df, _, _, preflight_exec_info = score_batch(
    dna_model,
    preflight_variant_df,
    gene_info=gene_info,
    gene_name=GENE_NAME,
    gene_id=GENE_ID,
    sequence_length_label=SEQUENCE_LENGTH,
    active_variant_scorers=active_variant_scorers,
    max_workers=1,
    progress_bar=False,
    retry_wait_seconds=RETRY_WAIT_SECONDS,
)

interval_genes_df = extract_interval_genes(preflight_full_scores_df)
interval_genes_df.to_csv(INTERVAL_GENES_PATH, index=False)
print(f'Wrote interval gene manifest: {INTERVAL_GENES_PATH}')
print(f'Genes returned in {SEQUENCE_LENGTH} TSS-centered interval: {len(interval_genes_df)}')

target_present = False
if not interval_genes_df.empty:
    gene_id_base = GENE_ID.split('.', 1)[0]
    target_present = (
        interval_genes_df['gene_name'].astype(str).str.upper().eq(GENE_NAME.upper()).any()
        or interval_genes_df['gene_id'].astype(str).eq(GENE_ID).any()
        or interval_genes_df['gene_id'].astype(str).str.replace(r'\\.\\d+$', '', regex=True).eq(gene_id_base).any()
    )

if not target_present:
    raise ValueError(
        f'Target gene {GENE_NAME} ({GENE_ID}) was not found in the AlphaGenome-returned interval gene list. '
        f'Inspect {INTERVAL_GENES_PATH} before running the full scoring step.'
    )

preflight_filtered_scores_df = filter_scores_to_target_context(
    preflight_full_scores_df,
    gene_name=GENE_NAME,
    gene_id=GENE_ID,
    data_source=TARGET_DATA_SOURCE,
    gtex_tissue=TARGET_GTEX_TISSUE,
)
print(
    f'Rows for {GENE_NAME} after {TARGET_DATA_SOURCE}/{TARGET_GTEX_TISSUE} filters on one preflight variant: '
    f'{len(preflight_filtered_scores_df)}'
)
if preflight_filtered_scores_df.empty:
    raise ValueError(
        f'No rows remained for {GENE_NAME} after filtering to data_source={TARGET_DATA_SOURCE!r} '
        f'and gtex_tissue={TARGET_GTEX_TISSUE!r}. Adjust filters before running the full scoring step.'
    )

interval_genes_df.head(20)


In [ ]:
initialize_batch_shard_dir(BATCH_SHARD_DIR)
aggregate_state = {}
variant_records = []
tss_radius = tss_radius_bp(SEQUENCE_LENGTH)
filtered_row_count = 0
written_shard_count = 0

print(
    f'Scoring {len(eligible_variants_df)} eligible variants '
    f'using score_variants(max_workers={SCORING_MAX_WORKERS}) and sequence length {SEQUENCE_LENGTH}'
)

for batch_idx, batch_df in enumerate(chunked(eligible_variants_df, BATCH_SIZE), start=1):
    print(f'Processing batch {batch_idx}: {len(batch_df)} variants')
    batch_full_scores_df, batch_scored_variants_df, batch_elapsed_seconds, batch_exec_info = score_batch(
        dna_model,
        batch_df,
        gene_info=gene_info,
        gene_name=GENE_NAME,
        gene_id=GENE_ID,
        sequence_length_label=SEQUENCE_LENGTH,
        active_variant_scorers=active_variant_scorers,
        max_workers=SCORING_MAX_WORKERS,
        progress_bar=False,
        retry_wait_seconds=RETRY_WAIT_SECONDS,
    )
    variant_records.extend(batch_scored_variants_df.to_dict('records'))

    filtered_batch_scores_df = filter_scores_to_target_context(
        batch_full_scores_df,
        gene_name=GENE_NAME,
        gene_id=GENE_ID,
        data_source=TARGET_DATA_SOURCE,
        gtex_tissue=TARGET_GTEX_TISSUE,
    )
    filtered_row_count += len(filtered_batch_scores_df)
    update_aggregate_state(aggregate_state, filtered_batch_scores_df)

    write_started = pd.Timestamp.utcnow()
    shard_path = write_filtered_batch_shard(filtered_batch_scores_df, BATCH_SHARD_DIR, batch_idx)
    write_elapsed_seconds = (pd.Timestamp.utcnow() - write_started).total_seconds()
    if shard_path is not None:
        written_shard_count += 1

    print(
        f'  API scoring: {batch_elapsed_seconds:.2f}s '
        f'({batch_elapsed_seconds / len(batch_df):.3f}s/variant); '
        f'parallel_used={batch_exec_info["used_parallel"]}; retries={batch_exec_info["retry_count"]}'
    )
    if batch_exec_info['last_retry_reason']:
        print('  Last retry reason: quota exceeded; waited and retried the same parallel batch.')
    print(
        f'  Batch write: {write_elapsed_seconds:.2f}s; '
        f'filtered_rows={len(filtered_batch_scores_df)}; '
        f'shard_written={shard_path is not None}'
    )

    del batch_full_scores_df, batch_scored_variants_df, filtered_batch_scores_df
    gc.collect()

scored_variants_df = pd.DataFrame(variant_records)

print('Scoring complete.')
print(f'Variant records: {len(scored_variants_df)}')
print(f'Filtered rows accumulated across shards: {filtered_row_count}')
print(f'Batch shard files written: {written_shard_count}')


In [ ]:
final_filtered_row_count = finalize_filtered_scores(BATCH_SHARD_DIR, FILTERED_SCORES_PATH)
aggregated_scores_df = finalize_aggregated_scores(aggregate_state)
aggregated_scores_df.to_csv(AGGREGATED_SCORES_PATH, index=False)

print(f'Wrote: {FILTERED_SCORES_PATH}')
print(f'Wrote: {AGGREGATED_SCORES_PATH}')
print(f'Final filtered score rows: {final_filtered_row_count}')
print(f'Aggregated variants: {len(aggregated_scores_df)}')
print(f'Variants scored successfully: {(scored_variants_df["score_status"] == "scored").sum() if not scored_variants_df.empty else 0}')
print(f'Variants with API errors: {(scored_variants_df["score_status"] == "api_error").sum() if not scored_variants_df.empty else 0}')

if aggregated_scores_df.empty:
    print('No aggregated scores were generated after filtering to the target context.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(aggregated_scores_df['alphagenome_raw_mean'], bins=40, color='#2f6c8f', edgecolor='white')
    ax.set_title(f'{GENE_NAME} AlphaGenome Mean Raw Score Distribution')
    ax.set_xlabel('Mean raw score')
    ax.set_ylabel('Variant count')
    fig.tight_layout()
    fig.savefig(ANNOTATION_HISTOGRAM_PATH, dpi=150)
    plt.show()
    plt.close(fig)
    print(f'Wrote: {ANNOTATION_HISTOGRAM_PATH}')

    trimmed_scores_df = aggregated_scores_df[
        ~aggregated_scores_df['alphagenome_raw_mean'].between(-0.002, 0.002, inclusive='both')
    ].copy()
    ignored_count = len(aggregated_scores_df) - len(trimmed_scores_df)
    print(f'Ignoring {ignored_count} variants with alphagenome_raw_mean in [-0.002, 0.002] for the trimmed histogram.')

    if trimmed_scores_df.empty:
        print('No variants remain after removing near-zero mean raw scores.')
    else:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(trimmed_scores_df['alphagenome_raw_mean'], bins=40, color='#b85c38', edgecolor='white')
        ax.set_title(f'{GENE_NAME} AlphaGenome Mean Raw Score Distribution (|score| > 0.002)')
        ax.set_xlabel('Mean raw score')
        ax.set_ylabel('Variant count')
        fig.tight_layout()
        fig.savefig(ANNOTATION_TRIMMED_HISTOGRAM_PATH, dpi=150)
        plt.show()
        plt.close(fig)
        print(f'Wrote: {ANNOTATION_TRIMMED_HISTOGRAM_PATH}')

with pd.option_context('display.max_columns', None, 'display.width', None):
    display(aggregated_scores_df.head())
